# ?? ??? ??: ?? ??? ??? ?? ?? (Deep Learning)

> **Why This Notebook?**
> ? ???? **Deep Learning (MLP)** ??? ???? ??? ?? ??? ?????.
> ?? ?? ????? ?? ???? **???(0?) ??**? **??? ????**? ??? ??? ??? ??? ?????.

**Analyst**: @Antigravity  
**Date**: 2026-01-07  
**Goal**: MLP ??? ???? F1-Score ?? ??? ?? ??.

## ?? 1. ?? ?? ? ????? (Setup & Imports)

- ?? ???? ?? Random Seed? ?????.
- ???? ?? `tensorflow`, ??? ??? ?? `pandas`, `sklearn`? ?????.

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing & Evaluation
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression

# Deep Learning
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# ??? ??
import platform
system_name = platform.system()
if system_name == 'Darwin': plt.rc('font', family='AppleGothic')
elif system_name == 'Windows': plt.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False
sns.set_palette("pastel")

# Random Seed
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("? Environment Ready.")

## ?? 2. ??? ?? ? ?? ?? (Obtain & Scrub)

### 2.1 ??? ??
???? ??? ??? ??? ?????.

In [ ]:
df = pd.read_csv('../../data/pima-indians-diabetes3.csv')
display(df.head())
print(df.info())

### 2.2 ???(Zero Value) ??
> **Logical Failure Check**: 
> `Glucose`(???), `BloodPressure`(??), `SkinThickness`(????), `Insulin`(???), `BMI`(?????)? ?????? 0? ? ? ????.
> ?????? ???? 0?? ??? ??? ????, ?? `NaN`?? ?? ? ??? ??? ??(Imputation)???.

In [ ]:
zero_cols = ['plasma', 'pressure', 'thickness', 'insulin', 'bmi']

# 0? NaN?? ??
df[zero_cols] = df[zero_cols].replace(0, np.nan)

# ??? ?? ??
print("?? Missing Values (Originally 0):")
print(df.isnull().sum())

### 2.3 ??? ?? (Strategic Imputation)
??? ??? ????? ?? ?? ?? **??/??? ??**? ?????.
- ???? ??? ??? ??? ?? `mean`(??)?? ???????. (SKILL ??? ?? SimpleImputer ??)

In [ ]:
imputer = SimpleImputer(strategy='mean')
df[zero_cols] = imputer.fit_transform(df[zero_cols])

print("? Imputation Complete.")
display(df.describe())

## ?? 3. ??? ??? ?? (EDA)

### 3.1 ???? ??
?? ??? ??? ??(`diabetes`)? ?? ??? ??? ?????.

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix')
plt.show()

## ?? 4. ??? ??? (Preprocessing)

### 4.1 ???? (Scaling)
> **Why StandardScaler?**
> ??? ??? ?? ???? ???? ?? ?????. `Insulin`? ?? ??? ?? `PedigreeFunction`? 0?? ?????.
> ?? ?? 0, ?? 1? ???? ?? ???? ?????.

In [ ]:
X = df.drop('diabetes', axis=1)
y = df['diabetes']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train/Test Split (Stratified to maintain class balance)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=SEED, stratify=y
)

print(f"Train Shape: {X_train.shape}, Test Shape: {X_test.shape}")

## ?? 5. ??? ??? (Deep Learning)

### 5.1 MLP ???? ??
- **Input**: 8 Features
- **Hidden 1**: 32 Units, ReLU
- **Dropout**: 0.2 (Overfitting ??)
- **Hidden 2**: 16 Units, ReLU
- **Output**: 1 Unit, Sigmoid (?? ??)

? ????? 768?? ?? ????, ??? ?? ??? ????? ?????.

In [ ]:
model = Sequential([
    Dense(32, activation='relu', input_shape=(8,)),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer=Adam(learning_rate=0.001), 
              loss='binary_crossentropy', 
              metrics=['accuracy'])

model.summary()

### 5.2 ?? ??
- **EarlyStopping**: ?? ??(`val_loss`)? ? ?? ???? ??? ??? ?? ?????.

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=0  # ?? ???
)

# ?? ?? ???
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss Curve')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.title('Accuracy Curve')
plt.legend()

plt.show()

## ?? 6. ?? ?? (Evaluation)

### 6.1 ?? ??
Accuracy ?? ??? F1-Score, ROC-AUC? ?????.

In [ ]:
y_pred_probs = model.predict(X_test)
y_pred = (y_pred_probs > 0.5).astype(int)

print("Classification Report:")
print(classification_report(y_test, y_pred))

f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_probs)

print(f"Final F1-Score: {f1:.4f}")
print(f"Final ROC-AUC: {auc:.4f}")

### 6.2 Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## ?? 7. ?? (Conclusion)

- ??? ??? StandardScaler? ??? ??? ?? ???? ??? ?????.
- ????? ?? ???? ??? '?? ?????? ?? ??'? ?????? ??? ? ????.
- ?? `XGBoost` ? ?? ?? ??? ??? ??? ???? ??? ????.